In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
train_dir = '/home/mohamed/autonom_ws/src/Action-Emotions_deep-learning-model/emotions/train'
test_dir = '/home/mohamed/autonom_ws/src/Action-Emotions_deep-learning-model/emotions/test'

img_size = (48, 48)
batch_size = 64

# ==========================================
# 2. DATA GENERATORS
# ==========================================

# A. Training Data Generator (Augmentation + Split)
# validation_split=0.2 means: "Use 20% of files in this folder for validation"
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # <--- CRITICAL: Splits train folder
)

# B. Validation Data Generator (No Augmentation + Split)
# We must use a separate instance so we don't rotate/flip the validation images.
# We use the SAME validation_split ratio so it knows which files to pick.
valid_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2  # <--- Matches the split above
)

# C. Test Data Generator (No Augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

# ==========================================
# 3. FLOW FROM DIRECTORY
# ==========================================

print("Loading Training Set:")
train_generator = train_datagen.flow_from_directory(
    train_dir,                # Point to Train Folder
    target_size=img_size,
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='sparse',
    subset='training',        # <--- Get the Training subset
    shuffle=True
)

print("Loading Validation Set:")
validation_generator = valid_datagen.flow_from_directory(
    train_dir,                # Point to Train Folder (Same as above)
    target_size=img_size,
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='sparse',
    subset='validation',      # <--- Get the Validation subset
    shuffle=False             # No need to shuffle validation typically
)

print("Loading Test Set:")
test_generator = test_datagen.flow_from_directory(
    test_dir,                 # Point to Test Folder
    target_size=img_size,
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='sparse',
    shuffle=False             # Never shuffle test data if you want to verify specific files
)

2025-12-15 15:37:29.198014: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/mohamed/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Loading Training Set:
Found 22968 images belonging to 7 classes.
Loading Validation Set:
Found 5741 images belonging to 7 classes.
Loading Test Set:
Found 7178 images belonging to 7 classes.


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

model = Sequential()

# ==========================================
# BLOCK 1: 64 Filters (Low-level features)
# ==========================================
# Two convolutions allow the model to learn "edges" and "curves" robustly
model.add(Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(48, 48, 1)))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.3)) 

# ==========================================
# BLOCK 2: 128 Filters (Mid-level features)
# ==========================================
# Learning shapes like "eyes", "mouths", "noses"
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.3))

# ==========================================
# BLOCK 3: 256 Filters (High-level features)
# ==========================================
# Learning facial structure and expressions
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

# ==========================================
# BLOCK 4: 512 Filters (Abstract features)
# ==========================================
# Learning subtle emotional nuances
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.5))

# ==========================================
# CLASSIFICATION HEAD
# ==========================================
model.add(Flatten())

# Dense Layer 1
model.add(Dense(1024, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

# Output Layer (7 Emotions)
model.add(Dense(7, activation='softmax'))

# ==========================================
# COMPILE
# ==========================================

model.summary()

/home/mohamed/.local/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1765805855.501772  109551 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 728 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1660 Ti, pci bus id: 0000:01:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 6, 6, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 6, 6, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 3, 3, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1024)           │     4,719,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1024)           │         4,09

 Total params: 9,418,951 (35.93 MB)

 Trainable params: 9,414,983 (35.92 MB)

 Non-trainable params: 3,968 (15.50 KB)

In [ ]:
import os

# Allow TensorFlow to use the driver's JIT if ptxas is missing
os.environ['XLA_FLAGS'] = '--xla_gpu_unsafe_fallback_to_driver_on_ptxas_not_found=true'
from tensorflow.keras.optimizers import Adam
# ==========================================
# 4. COMPILE AND TRAIN (UPDATED FOR SPARSE)
# ==========================================
from tensorflow.keras.callbacks import EarlyStopping

# 1. Define the Early Stopping Callback
early_stopping = EarlyStopping(
    monitor='val_loss',         # Watch the validation loss
    patience=8,                 # Stop if it doesn't improve for 8 epochs
    min_delta=0.001,            # Minimum change to qualify as an improvement
    restore_best_weights=True,  # IMPORTANT: Revert to the best model found, not the last one
    verbose=1
)

# 2. Add it to your training loop

model.compile(
    optimizer=Adam(learning_rate=0.0005), # Slightly higher LR for this larger model
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


epochs = 150 

print(f"Starting training for {epochs} epochs...")

history = model.fit(
    train_generator,
    epochs=epochs,                 # Set a high max, early stopping will cut it short
    validation_data=validation_generator,
    validation_steps=validation_generator.n // validation_generator.batch_size,
    callbacks=[early_stopping] 
)

# ==========================================
# 5. SAVE THE MODEL
# ==========================================
model.save('emotion_model_sparse_vgg_deep.keras')
print("Model saved as 'emotion_model_sparse_vgg_deep.keras'")

Starting training for 150 epochs...
Epoch 1/150


2025-12-15 15:38:22.165864: I external/local_xla/xla/service/service.cc:163] XLA service 0x72fc88003b10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-15 15:38:22.165879: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5
2025-12-15 15:38:22.264973: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-15 15:38:22.965268: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2025-12-15 15:38:23.626884: I external/local_xla/xla/service/gpu/autotuning/conv_algorithm_picker.cc:546] Omitted potentially buggy algorithm eng14{k25=2} for conv (f32[64,64,48,48]{3,2,1,0}, u8[0]{0}) custom-call(f32[64,1,48,48]{3,2,1,0}, f32[64,1,3,3]{3,2,1,0}, f32[64]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_

KeyboardInterrupt: 

In [ ]:
import matplotlib
# CRITICAL: This line must be BEFORE you import pyplot
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ==========================================
# 6. VISUALIZATION & LOGGING
# ==========================================

# A. Save the raw numbers to CSV
hist_df = pd.DataFrame(history.history)
hist_df.to_csv('training_history2.csv', index=False)
print("Training history saved to 'training_history.csv'")

# B. Plot Accuracy and Loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy
ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend(loc='lower right')
ax1.grid(True)

# Plot 2: Loss
ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend(loc='upper right')
ax2.grid(True)

# Save the training graphs
plt.savefig('training_graphs.png', dpi=300)
print("Graphs saved as 'training_graphs.png'")
plt.close() # Close to free up memory

# ==========================================
# 7. CONFUSION MATRIX
# ==========================================
print("\nGenerating Confusion Matrix...")

# 1. Get Predictions
# Important: Reset generator to start from the beginning
validation_generator.reset() 

# Predict on all validation data
preds = model.predict(validation_generator, verbose=1)
y_pred = np.argmax(preds, axis=1) # Convert probabilities to class labels (0, 1, 2...)
y_true = validation_generator.classes # True labels from the folder structure

# 2. Compute Matrix
cm = confusion_matrix(y_true, y_pred)
class_labels = list(validation_generator.class_indices.keys()) # e.g. ['angry', 'happy', ...]

# 3. Plot Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_labels, 
            yticklabels=class_labels)

plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# 4. Save Confusion Matrix
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
print("Confusion Matrix saved as 'confusion_matrix.png'")
plt.close()

# Optional: Print a text report for precision/recall details
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))